# Baseline: single tenant

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

power_cap = 1800
gpu_counts = [1, 2, 3, 4]
total_gpus = 4
idle_power = 70  # W per GPU

# Choose optimization objective: 'energy' or 'edp'
optimization_objective = 'energy'
optimization_objective = 'edp'

spec_benchmarks = ['pot3d', 'minisweep', 'lbm', 'cloverleaf', 'tealeaf', 'miniweather','hpgmg']
ml_benchmarks = ['bert', 'gpt2', 'resnet50']

all_configs = [
    (Path('H100/spec_power_motif'), spec_benchmarks),
    (Path('H100/ml_power_motif'), ml_benchmarks),
]

# Step 1: Collect energy and runtime for all apps at all GPU counts (1800W)
app_data = {}  # app -> {gpu_count: {energy, runtime}}

for base_dir, benchmarks in all_configs:
    for bench in benchmarks:
        app_data[bench] = {}
        for gpu_count in gpu_counts:
            csv_file = base_dir / bench / f"{power_cap}_{gpu_count}_gpu_metrics.csv"
            if not csv_file.exists():
                continue

            df = pd.read_csv(csv_file)
            execution_time = df['Time (s)'].iloc[-1] - df['Time (s)'].iloc[0]

            runtime_file = base_dir / bench / 'runtime.csv'
            if runtime_file.exists():
                df_runtime = pd.read_csv(runtime_file)
                match = df_runtime[(df_runtime['power_cap'] == power_cap) &
                                   (df_runtime['gpu_count'] == gpu_count)]
                if len(match) > 0:
                    execution_time = match.iloc[0]['runtime_seconds']

            total_energy = 0.0
            for i in range(gpu_count):
                power_col = f'GPU{i}_Power (W)'
                if power_col in df.columns:
                    avg_power = df[power_col].mean()
                    total_energy += avg_power * execution_time

            app_data[bench][gpu_count] = {
                'energy': total_energy,
                'runtime': execution_time,
                'edp': total_energy * execution_time,
            }

# Step 2: For each app, find the GPU count that minimizes the chosen objective
best_config = {}
for app, gpu_configs in app_data.items():
    best_gpu = min(gpu_configs, key=lambda g: gpu_configs[g][optimization_objective])
    best_config[app] = {
        'gpu_count': best_gpu,
        'energy': gpu_configs[best_gpu]['energy'],
        'runtime': gpu_configs[best_gpu]['runtime'],
        'edp': gpu_configs[best_gpu]['edp'],
    }

# Step 3: Sequential execution — total system energy including idle GPUs
obj_label = 'Energy' if optimization_objective == 'energy' else 'EDP'
print(f"Baseline: Sequential execution (no co-scheduling), 4 H100 GPUs")
print(f"Optimization objective: least {obj_label}")
print("=" * 80)
print(f"{'App':<15} {'Best #GPUs':>10} {'Runtime (s)':>12} {'App Energy (J)':>15} "
      f"{'Idle Energy (J)':>16} {'Total Energy (J)':>16}")
print("-" * 80)

total_system_energy = 0.0
total_time = 0.0

for app, cfg in best_config.items():
    idle_gpus = total_gpus - cfg['gpu_count']
    idle_energy = idle_gpus * idle_power * cfg['runtime']
    system_energy = cfg['energy'] + idle_energy
    total_system_energy += system_energy
    total_time += cfg['runtime']

    print(f"{app:<15} {cfg['gpu_count']:>10} {cfg['runtime']:>12.2f} {cfg['energy']:>15.2f} "
          f"{idle_energy:>16.2f} {system_energy:>16.2f}")

print("-" * 80)
print(f"{'TOTAL':<15} {'':<10} {total_time:>12.2f} {'':<15} {'':<16} {total_system_energy:>16.2f}")
print(f"\nTotal sequential time:   {total_time:.2f} s")
print(f"Total system energy:     {total_system_energy:.2f} J ({total_system_energy/1000:.2f} kJ)")

Baseline: Sequential execution (no co-scheduling), 4 H100 GPUs
Optimization objective: least EDP
App             Best #GPUs  Runtime (s)  App Energy (J)  Idle Energy (J) Total Energy (J)
--------------------------------------------------------------------------------
pot3d                    1       133.09        35775.81         27948.07         63723.88
minisweep                4        43.61        23463.64             0.00         23463.64
lbm                      4        18.82        16921.61             0.00         16921.61
cloverleaf               4        16.83        22939.84             0.00         22939.84
tealeaf                  4        16.21        23193.73             0.00         23193.73
miniweather              1        57.01        27320.54         11971.35         39291.89
hpgmg                    2        10.78         3474.03          1509.00          4983.03
bert                     2        26.42        16495.24          3698.82         20194.05
gpt2        

# Co-Schedule

In [4]:
# Predict optimal GPU count (for least EDP) using dram_sum + sm_sum as runtime proxy
import math

PROFILE_SECONDS = 15.0

def extract_metrics_from_csv(csv_path, gpu_count):
    """Extract GPU performance metrics from first 15s of a gpu_metrics CSV."""
    df = pd.read_csv(csv_path)
    if len(df) < 2:
        return None

    if 'GPU0_DRAM_Active' in df.columns:
        df = df[df['GPU0_DRAM_Active'] != 0.0]
    if len(df) == 0:
        return None

    t0 = df['Time (s)'].min()
    df = df[df['Time (s)'] <= t0 + PROFILE_SECONDS]
    if len(df) == 0:
        return None

    active = [i for i in range(gpu_count) if f'GPU{i}_Power (W)' in df.columns]
    if not active:
        return None
    n_active = len(active)

    p_avg, dr_avg, sm_avg = {}, {}, {}
    fp16_avg, fp32_avg, fp64_avg = {}, {}, {}

    for gid in active:
        p_avg[gid] = df[f'GPU{gid}_Power (W)'].mean()
        sm_avg[gid] = df[f'GPU{gid}_SM_Clock (MHz)'].mean() if f'GPU{gid}_SM_Clock (MHz)' in df.columns else 0
        dr_avg[gid] = df[f'GPU{gid}_DRAM_Active'].mean() if f'GPU{gid}_DRAM_Active' in df.columns else 0
        fp16_avg[gid] = df[f'GPU{gid}_FP16_Active'].mean() if f'GPU{gid}_FP16_Active' in df.columns else 0
        fp32_avg[gid] = df[f'GPU{gid}_FP32_Active'].mean() if f'GPU{gid}_FP32_Active' in df.columns else 0
        fp64_avg[gid] = df[f'GPU{gid}_FP64_Active'].mean() if f'GPU{gid}_FP64_Active' in df.columns else 0

    def safe(v):
        return 0.0 if (v is None or (isinstance(v, float) and math.isnan(v))) else v

    dr_list = [safe(dr_avg[g]) for g in active]
    sm_list = [safe(sm_avg[g]) for g in active]
    fp_list = [safe(fp16_avg[g]) + safe(fp32_avg[g]) + safe(fp64_avg[g]) for g in active]

    return {
        'avg_power': safe(np.mean([safe(p_avg[g]) for g in active])),
        'dram_sum': sum(dr_list),
        'sm_sum': sum(sm_list),
        'fp_sum': sum(fp_list),
        'dram_min_scaled': min(dr_list) * n_active if n_active else 0,
        'sm_min_scaled': min(sm_list) * n_active if n_active else 0,
        'fp_min_scaled': min(fp_list) * n_active if n_active else 0,
    }

# Step 1: Extract metrics for all apps at all GPU counts (1800W) and write to txt
txt_lines = []
all_app_metrics = {}

for base_dir, benchmarks in all_configs:
    suite = 'SPEC' if 'spec' in str(base_dir) else 'ML'
    for app in benchmarks:
        all_app_metrics[app] = {}
        app_lines = []
        for gc in gpu_counts:
            csv_file = base_dir / app / f"{power_cap}_{gc}_gpu_metrics.csv"
            if not csv_file.exists():
                continue
            metrics = extract_metrics_from_csv(csv_file, gc)
            if metrics is None:
                continue
            all_app_metrics[app][gc] = metrics
            perf = app_data[app][gc]['runtime'] if app in app_data and gc in app_data[app] else float('nan')
            app_lines.append((gc, perf, metrics))

        if app_lines:
            app_lines.sort(key=lambda x: app_data[app][x[0]]['edp'] if app in app_data and x[0] in app_data[app] else float('inf'))
            edp_rank = [x[0] for x in app_lines]
            txt_lines.append(f"\n===== {suite}/{app} =====")
            txt_lines.append(f"cap={power_cap}\tedp_rank={edp_rank}")
            txt_lines.append("gpu_count\tperformance\tavg_power\tdram_sum\tsm_sum\tfp_sum\tdram_min_scaled\tsm_min_scaled\tfp_min_scaled")
            for gc, perf, m in app_lines:
                txt_lines.append(
                    f"{gc}\t{perf:.2f}\t{m['avg_power']:.2f}\t{m['dram_sum']:.2f}\t{m['sm_sum']:.2f}\t{m['fp_sum']:.2f}\t"
                    f"{m['dram_min_scaled']:.2f}\t{m['sm_min_scaled']:.2f}\t{m['fp_min_scaled']:.2f}"
                )

txt_output = '\n'.join(txt_lines)
txt_path = Path('H100/edp_metrics.txt')
txt_path.write_text(txt_output)
print(txt_output)
print(f"\n[written] {txt_path}")

# Step 2: Grid search for best (w, alpha) in:
#   activity = dram_sum + w * sm_sum
#   norm_rt  = (max_activity / activity) ^ alpha
#   norm_edp = norm_rt * avg_power * gpu_count
#   5% tolerance: prefer smaller GPU count if within 5%

all_apps = [app for app in all_app_metrics if len(all_app_metrics[app]) == len(gpu_counts)]

# Compute true best GPU count (with 5% tolerance) for each app
true_best = {}
for app in all_apps:
    true_edps = {gc: app_data[app][gc]['edp'] for gc in gpu_counts}
    best_true_edp = min(true_edps.values())
    true_gc = min(true_edps, key=lambda gc: true_edps[gc])
    for gc_c in sorted(gpu_counts):
        if gc_c >= true_gc:
            break
        if true_edps[gc_c] <= best_true_edp * 1.05:
            true_gc = gc_c
            break
    true_best[app] = true_gc

def predict_with_params(w, alpha):
    """Return (accuracy, predictions_dict) for given w, alpha."""
    correct = 0
    preds = {}
    for app in all_apps:
        dram_s = {gc: all_app_metrics[app][gc]['dram_sum'] for gc in gpu_counts}
        sm_s = {gc: all_app_metrics[app][gc]['sm_sum'] for gc in gpu_counts}
        combined = {gc: dram_s[gc] + w * sm_s[gc] for gc in gpu_counts}
        max_comb = max(combined.values())

        best_edp = float('inf')
        best_gc = None
        gc_edps = {}
        for gc in gpu_counts:
            if combined[gc] > 1e-9:
                norm_rt = (max_comb / combined[gc]) ** alpha
            else:
                norm_rt = float('inf')
            norm_edp = norm_rt * all_app_metrics[app][gc]['avg_power'] * gc
            gc_edps[gc] = norm_edp
            if norm_edp < best_edp:
                best_edp = norm_edp
                best_gc = gc

        for gc_candidate in sorted(gpu_counts):
            if gc_candidate >= best_gc:
                break
            if gc_edps[gc_candidate] <= best_edp * 1:
                best_gc = gc_candidate
                break

        preds[app] = best_gc
        if best_gc == true_best[app]:
            correct += 1
    return correct, preds

# Grid search
print("\n\nSearching for best (w, alpha) coefficients...")
print("=" * 60)

best_accuracy = 0
best_params = (0, 1.0)
results = []

for w in np.arange(0, 0.0005, 0.000005):
    for alpha in np.arange(0.5, 2.5, 0.05):
        acc, _ = predict_with_params(w, alpha)
        if acc > best_accuracy:
            best_accuracy = acc
            best_params = (w, alpha)
            results.append((w, alpha, acc))
        if acc == len(all_apps):
            break
    if best_accuracy == len(all_apps):
        break

w_best, alpha_best = best_params
print(f"Best params: w={w_best:.6f}, alpha={alpha_best:.2f}")
print(f"Best accuracy: {best_accuracy}/{len(all_apps)}")

# Show top results
print("\nTop parameter combos:")
for w, alpha, acc in sorted(results, key=lambda x: -x[2])[:10]:
    print(f"  w={w:.6f}, alpha={alpha:.2f} -> {acc}/{len(all_apps)}")

# Step 3: Use best params for final prediction
print(f"\n\nFinal prediction with w={w_best:.6f}, alpha={alpha_best:.2f} (5% tolerance)")
print("=" * 100)
print(f"{'App':<15} {'GPU':>4} {'dram_sum':>10} {'sm_sum':>10} {'activity':>10} {'avg_power':>10} {'norm_rt':>8} {'true_norm_rt':>12} {'norm_EDP':>12}")
print("-" * 100)

correct = 0
predicted_config = {}

for app in all_apps:
    dram_sums = {gc: all_app_metrics[app][gc]['dram_sum'] for gc in gpu_counts}
    sm_sums = {gc: all_app_metrics[app][gc]['sm_sum'] for gc in gpu_counts}
    combined = {gc: dram_sums[gc] + w_best * sm_sums[gc] for gc in gpu_counts}
    max_comb = max(combined.values())

    runtimes = {gc: app_data[app][gc]['runtime'] for gc in gpu_counts}
    min_runtime = min(runtimes.values())

    best_norm_edp = float('inf')
    best_gc = None
    rows = []

    for gc in gpu_counts:
        avg_pow = all_app_metrics[app][gc]['avg_power']
        if combined[gc] > 1e-9:
            norm_rt = (max_comb / combined[gc]) ** alpha_best
        else:
            norm_rt = float('inf')
        true_norm_rt = runtimes[gc] / min_runtime
        norm_edp = norm_rt * avg_pow * gc
        rows.append((gc, dram_sums[gc], sm_sums[gc], combined[gc], avg_pow, norm_rt, true_norm_rt, norm_edp))
        if norm_edp < best_norm_edp:
            best_norm_edp = norm_edp
            best_gc = gc

    for gc_candidate in sorted(gpu_counts):
        if gc_candidate >= best_gc:
            break
        candidate_edp = [nedp for (g, _, _, _, _, _, _, nedp) in rows if g == gc_candidate][0]
        if candidate_edp <= best_norm_edp * 1:
            best_gc = gc_candidate
            break

    predicted_config[app] = {
        'gpu_count': best_gc,
        'runtime': app_data[app][best_gc]['runtime'],
        'energy': app_data[app][best_gc]['energy'],
        'edp': app_data[app][best_gc]['edp'],
    }

    true_gc = true_best[app]
    status = "OK" if best_gc == true_gc else "MISS"
    correct += (best_gc == true_gc)

    for i, (gc, dram, sm, act, avg_pow, norm_rt, true_norm_rt, norm_edp) in enumerate(rows):
        marker = " <--" if gc == best_gc else ""
        app_label = app if i == 0 else ""
        print(f"{app_label:<15} {gc:>4} {dram:>10.4f} {sm:>10.2f} {act:>10.4f} {avg_pow:>10.2f} {norm_rt:>8.2f} {true_norm_rt:>12.2f} {norm_edp:>12.2f}{marker}")

    print(f"{'':>15} pred={best_gc}, true={true_gc}  {status}")
    print()

print(f"Accuracy: {correct}/{len(all_apps)} ({100*correct/len(all_apps):.1f}%)")


===== SPEC/pot3d =====
cap=1800	edp_rank=[1, 2, 3, 4]
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum	dram_min_scaled	sm_min_scaled	fp_min_scaled
1	133.09	266.32	0.21	1980.00	0.00	0.21	1980.00	0.00
2	109.34	201.76	0.25	3960.00	0.01	0.25	3960.00	0.01
3	104.33	173.49	0.28	5940.00	0.01	0.28	5940.00	0.01
4	102.72	158.43	0.28	7920.00	0.01	0.28	7920.00	0.01

===== SPEC/minisweep =====
cap=1800	edp_rank=[4, 3, 2, 1]
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum	dram_min_scaled	sm_min_scaled	fp_min_scaled
4	43.61	141.45	0.09	7920.00	0.04	0.09	7920.00	0.04
3	57.47	141.75	0.06	5940.00	0.03	0.06	5940.00	0.03
2	77.25	149.37	0.04	3960.00	0.02	0.04	3960.00	0.02
1	140.82	166.53	0.03	1980.00	0.01	0.03	1980.00	0.01

===== SPEC/lbm =====
cap=1800	edp_rank=[4, 3, 2, 1]
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum	dram_min_scaled	sm_min_scaled	fp_min_scaled
4	18.82	285.17	0.52	7920.00	0.71	0.51	7920.00	0.70
3	21.98	295.44	0.39	5940.00	0.54	0.39	5940.00	0.54
2	29.83	308.18	0.27	3

In [5]:
# Co-scheduling with FCFS policy (max 2 apps concurrently) using PREDICTED GPU counts
job_queue = ['pot3d', 'minisweep', 'lbm', 'cloverleaf', 'tealeaf', 'miniweather',
             'bert', 'gpt2', 'resnet50','hpgmg']

job_queue = ['pot3d', 'minisweep', 'lbm', 'cloverleaf', 'tealeaf', 'miniweather',
             'bert', 'gpt2', 'resnet50','hpgmg']

max_concurrent_apps = 2

pending = list(job_queue)
running = []
current_time = 0.0
total_system_energy = 0.0
schedule_log = []

print(f"Co-scheduling (FCFS, max {max_concurrent_apps} apps), 4 H100 GPUs, 1800W cap")
print(f"Using PREDICTED GPU counts (dram_sum proxy)")
print("=" * 80)

while pending or running:
    gpus_used = sum(r['gpu_count'] for r in running)
    gpus_free = total_gpus - gpus_used

    scheduled_this_round = True
    while scheduled_this_round and pending and len(running) < max_concurrent_apps:
        scheduled_this_round = False
        for app in list(pending):
            needed = predicted_config[app]['gpu_count']
            if needed <= gpus_free:
                runtime = predicted_config[app]['runtime']
                energy = predicted_config[app]['energy']
                end_time = current_time + runtime
                energy_rate = energy / runtime

                running.append({
                    'app': app,
                    'gpu_count': needed,
                    'energy_rate': energy_rate,
                    'start_time': current_time,
                    'end_time': end_time,
                    'runtime': runtime,
                    'energy': energy,
                })
                pending.remove(app)
                gpus_free -= needed
                scheduled_this_round = True
                print(f"  t={current_time:8.2f}s | START {app:<15} | {needed} GPUs | "
                      f"ends at t={end_time:.2f}s | free GPUs: {gpus_free}")
                break

    if not running:
        break

    next_end = min(r['end_time'] for r in running)
    dt = next_end - current_time

    gpus_used = sum(r['gpu_count'] for r in running)
    idle_gpus = total_gpus - gpus_used

    for r in running:
        remaining = min(r['end_time'], next_end) - current_time
        total_system_energy += r['energy_rate'] * remaining

    total_system_energy += idle_gpus * idle_power * dt

    current_time = next_end
    finished = [r for r in running if r['end_time'] <= current_time]
    for r in finished:
        schedule_log.append(r)
        running.remove(r)
        print(f"  t={current_time:8.2f}s | END   {r['app']:<15} | freed {r['gpu_count']} GPUs")

total_time = current_time

print("-" * 80)
print(f"\nSchedule summary:")
print(f"{'App':<15} {'#GPUs':>6} {'Start (s)':>10} {'End (s)':>10} {'Runtime (s)':>12} {'App Energy (J)':>15}")
print("-" * 70)
for r in schedule_log:
    print(f"{r['app']:<15} {r['gpu_count']:>6} {r['start_time']:>10.2f} {r['end_time']:>10.2f} "
          f"{r['runtime']:>12.2f} {r['energy']:>15.2f}")

print("-" * 70)
print(f"\nTotal makespan:          {total_time:.2f} s")
print(f"Total system energy:     {total_system_energy:.2f} J ({total_system_energy/1000:.2f} kJ)")

Co-scheduling (FCFS, max 2 apps), 4 H100 GPUs, 1800W cap
Using PREDICTED GPU counts (dram_sum proxy)
  t=    0.00s | START pot3d           | 1 GPUs | ends at t=133.09s | free GPUs: 3
  t=    0.00s | START miniweather     | 1 GPUs | ends at t=57.01s | free GPUs: 2
  t=   57.01s | END   miniweather     | freed 1 GPUs
  t=   57.01s | START gpt2            | 3 GPUs | ends at t=71.81s | free GPUs: 0
  t=   71.81s | END   gpt2            | freed 3 GPUs
  t=   71.81s | START resnet50        | 3 GPUs | ends at t=92.62s | free GPUs: 0
  t=   92.62s | END   resnet50        | freed 3 GPUs
  t=   92.62s | START hpgmg           | 2 GPUs | ends at t=103.40s | free GPUs: 1
  t=  103.40s | END   hpgmg           | freed 2 GPUs
  t=  133.09s | END   pot3d           | freed 1 GPUs
  t=  133.09s | START minisweep       | 4 GPUs | ends at t=176.70s | free GPUs: 0
  t=  176.70s | END   minisweep       | freed 4 GPUs
  t=  176.70s | START lbm             | 4 GPUs | ends at t=195.52s | free GPUs: 0
  t=  195.